# Milestone 3: Baseline Model

**Dataset:** House Sales in King County, USA (the engineered data from Milestone 2)

**Goal:** Train my first Machine Learning model. A baseline is a simple model. It tells me if the data can predict the target before I try harder models.

**Target variable:** `price`

## Step 1: Choose the right model

I want to predict `price`. Price is a **number**, so this is a **regression** problem.

- For a number target, the baseline model is **Linear Regression**.
- Logistic Regression is for category targets (like pass/fail). It is not right for price.

I will use these metrics for regression:
- **Mean Squared Error (MSE)** and **RMSE**: how far the predictions are from the real prices. Lower is better.
- **R2 score**: how much of the price change the model explains. 1 is perfect. 0 is the same as guessing the average.

Accuracy and the confusion matrix are for category targets. I do not use them here.

## Step 2: Import libraries and load the data

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(r"C:\data-science-internship\data\kc_house_data.csv\kc_house_data.csv")
print(df.shape)
df.head()

(21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


## Step 3: Separate the features (X) from the target (y)

- `y` is the column I want to predict: `price`.
- `X` is every other column.

In [11]:
# Convert the date column into a datetime format
df['date'] = pd.to_datetime(
    df['date'],
    format='%Y%m%dT%H%M%S'
)

# Extract useful numerical features from the date
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month

# Separate features and target
# Exclude price (target), id (identifier), and original date (text/datetime)
X = df.drop(columns=['price', 'id', 'date'])
y = df['price']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Non-numeric columns:",
      X.select_dtypes(exclude='number').columns.tolist())

X shape: (21613, 20)
y shape: (21613,)
Non-numeric columns: []


## Step 4: Split into training and testing sets

- 80% of the data is for training. The model learns from it.
- 20% of the data is for testing. The model has never seen it. This shows how the model does on new data.
- `random_state=42` gives the same split every time I run the notebook.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 17290
Testing rows:  4323


## Step 5: Train the model

I make a Linear Regression model. Then I train it with `fit()` on the training data only.

In [13]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## Step 6: Make predictions on the test set

In [14]:
y_pred = model.predict(X_test)

# Compare the first 5 predictions with the real prices
pd.DataFrame({'Real Price': y_test.values[:5], 'Predicted Price': y_pred[:5].round(0)})

,Real Price,Predicted Price
0,365000.0,454653.0
1,865000.0,744707.0
2,1038000.0,1228559.0
3,1490000.0,1683790.0
4,711000.0,723988.0


## Step 7: Evaluate the model

I calculate MSE, RMSE and R2 on the **test set**.

- RMSE is the square root of MSE. It is in dollars, so it is easier to understand.

In [15]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:,.0f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.0f}")
print(f"R2 Score: {r2:.3f}")

Mean Squared Error (MSE): 44,959,335,429
Root Mean Squared Error (RMSE): $212,036
R2 Score: 0.703


## Step 8: Is this baseline good or bad?

To judge the model, I compare it with **just guessing**. The simplest guess is to predict the average training price for every house. I calculate the error of this guess on the same test set.

In [16]:
guess = np.full(len(y_test), y_train.mean())

guess_rmse = np.sqrt(mean_squared_error(y_test, guess))
guess_r2 = r2_score(y_test, guess)

print(f"Guessing the average -> RMSE: ${guess_rmse:,.0f}, R2: {guess_r2:.3f}")
print(f"Linear Regression    -> RMSE: ${rmse:,.0f}, R2: {r2:.3f}")

improvement = (guess_rmse - rmse) / guess_rmse * 100
print(f"The model's error is {improvement:.0f}% lower than guessing.")

Guessing the average -> RMSE: $388,987, R2: -0.001
Linear Regression    -> RMSE: $212,036, R2: 0.703
The model's error is 45% lower than guessing.


### My explanation

**How to read the numbers:**
- Guessing the average always gives an R2 of about 0. This is the minimum I must beat.
- If the model has a much higher R2 and a much lower RMSE than guessing, the baseline is **good**. It means the features have real power to predict price.
- If the numbers are close to guessing, the baseline is **bad**. It means the features or the model are not enough.

**My conclusion:**
Compare the two lines printed above. The Linear Regression model has a much lower error than guessing the average, and its R2 is much higher than 0. So this baseline is **good for a first model**. The data clearly has predictive power.

**Why it is not perfect:**
- House prices have a few very high values. A straight line does not fit these well.
- Linear Regression cannot learn complex patterns. For example, price may not go up in a straight line with size.
- The RMSE is still large in dollars. A person buying a house would want a smaller error.

**Next step:** I can try stronger models and compare them with this baseline. If they cannot beat it, they are not worth the extra work.

Note: In Milestone 2, I fitted the scaler on all the data. This is simple, but it lets a little test information into training. It is a small effect for this baseline.

## Summary

- The target `price` is a number, so I used **Linear Regression** and the regression metrics (MSE, RMSE and R2).
- I split the data 80/20 and tested on unseen data.
- I compared the model with the guess-the-average baseline.
- The model beats guessing, so the data can predict price.